# SignalFrame on Colab

Measures a short clip, describes what it measured, and helps you turn that into an experiment
you can run. **It does not predict how an audience will respond**, and no behavioral head is
installed — so no probability appears anywhere in this notebook.

Run the cells in order. The last one prints a public URL you can open from any browser.

---

## What runs here

Every lane has a runtime that works on Colab. The three that used to be Apple-silicon only —
the transcript lane, the keyframe description lane, and the local insight model — now each
have a portable `torch` backend that runs on an NVIDIA GPU or on CPU.

| Lane | On Colab | Notes |
| --- | --- | --- |
| Media metadata, measured audio | **Yes** | Needs only `ffmpeg` |
| Hook readout, recut, variants, experiments | **Yes** | Pure measurement, no model |
| On-screen text (OCR) | **Yes** | `pytesseract` engine; Apple Vision is macOS-only |
| Transcript (ASR) | **Yes, on GPU** | `transformers` backend, `openai/whisper-large-v3-turbo` |
| NanoLLaVA keyframes | **Yes, on GPU** | `transformers` backend, `qnguyen3/nanoLLaVA-1.5` |
| Insight / Hook Doctor | **Yes** | `torch-local` on GPU, or the remote Anthropic provider |
| V-JEPA 2.1, AST AudioSet | **Yes, with artifacts** | Pinned checkpoints, hash-verified before load |
| TRIBE v2 cortical | **Yes, with gated access** | Colab runs the full trimodal path, not the macOS vision-only ablation |

Two honest caveats, because they will shape what you see:

**Every model lane stays unavailable until you pin its revision** to a full 40-character
commit SHA. That is deliberate, not an oversight: a mutable name like `main` can change
under you, which would make a result impossible to reproduce. Step 4 is where you set them.
Leave them blank and the measurement lanes still work.

**TPU is not supported by any lane.** These are torch models with custom vision towers;
they run on CUDA or CPU. A device request naming a TPU is refused by name rather than
silently ignored, so you will see a reason rather than a mysterious hang. Use a **GPU**
runtime.

## 1. Check what hardware you were given

In [ ]:
import subprocess, sys, platform

print("Python :", sys.version.split()[0], "on", platform.platform())
try:
    print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                          "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
          or "No GPU reported")
except FileNotFoundError:
    print("No GPU on this runtime. Everything except the cortical lane still works;")
    print("set Runtime > Change runtime type > GPU if you intend to install TRIBE v2.")

## 2. Get the code

In [ ]:
import os, subprocess

REPO = "https://github.com/Kayariyan28/Cognitive-Hook-Predictor.git"
BRANCH = "claude/updates-cagd0r"
ROOT = "/content/Cognitive-Hook-Predictor"

if not os.path.isdir(ROOT):
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO, ROOT], check=True)
else:
    subprocess.run(["git", "-C", ROOT, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", ROOT, "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", ROOT, "pull", "--ff-only"], check=True)

os.chdir(ROOT)
print("HEAD:", subprocess.run(["git", "rev-parse", "--short", "HEAD"],
                              capture_output=True, text=True).stdout.strip())

## 3. Install dependencies

Three groups: the model-free backend, the portable torch backends for the transcript,
keyframe and insight lanes, and Gradio for the interface. `ffmpeg` is what the
measured-audio branch needs; `tesseract-ocr` is what the on-screen-text engine needs.

Colab already ships a working `torch` built against its own CUDA, so
`requirements-portable.txt` deliberately does not pin one and will not replace it.

In [ ]:
!apt-get -qq update && apt-get -qq install -y ffmpeg tesseract-ocr > /dev/null
!pip install -q -r backend/requirements-local.txt
!pip install -q -r backend/requirements-portable.txt
!pip install -q gradio

import shutil, importlib
for binary in ("ffmpeg", "ffprobe", "tesseract"):
    print(f"{binary:<12}", shutil.which(binary) or "MISSING")
for module in ("torch", "transformers"):
    print(f"{module:<12}", importlib.import_module(module).__version__)

import torch
print("cuda        ", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NOT AVAILABLE")

## 4. Configure

This selects the portable backends and pins each model. **Pinning is what makes a lane
run at all** — leave a revision blank and that lane reports itself unavailable with a
reason, which is the design rather than a failure.

Look up a commit SHA on the model's Hugging Face page, under *Files and versions*. The
model repositories are:

- transcript — `openai/whisper-large-v3-turbo`
- keyframes — `qnguyen3/nanoLLaVA-1.5`
- insight — `Qwen/Qwen2.5-7B-Instruct` (or leave the insight lane on the remote provider)

`ANTHROPIC_API_KEY` is an alternative to running the insight model locally. Only derived
JSON evidence is ever sent — never your video, audio, frames or tensors.

In [ ]:
import os

# --- device ---------------------------------------------------------------
os.environ["SIGNALFRAME_TORCH_DEVICE"] = "auto"   # auto | cuda | mps | cpu
os.environ["SIGNALFRAME_TORCH_DTYPE"] = "auto"    # auto | float32 | float16 | bfloat16

# --- select the portable backends ----------------------------------------
os.environ["INSIGHT_ASR_BACKEND"] = "transformers"
os.environ["FORECAST_NANOLLAVA_BACKEND"] = "transformers"
os.environ["INSIGHT_OCR_ENGINE"] = "pytesseract"
os.environ["INSIGHT_COMPARATIVE_MINIMUM_CLIPS"] = "5"

# --- pin each model to an immutable commit -------------------------------
ASR_REVISION      = ""   # openai/whisper-large-v3-turbo
NANOLLAVA_REVISION = ""  # qnguyen3/nanoLLaVA-1.5
INSIGHT_REVISION  = ""   # Qwen/Qwen2.5-7B-Instruct

ANTHROPIC_API_KEY = ""   # alternative to running the insight model locally
HF_TOKEN = ""            # only needed for the gated TRIBE v2 lane

if ASR_REVISION.strip():
    os.environ["INSIGHT_ASR_MODEL"] = "openai/whisper-large-v3-turbo"
    os.environ["INSIGHT_ASR_MODEL_REVISION"] = ASR_REVISION.strip()

if NANOLLAVA_REVISION.strip():
    os.environ["FORECAST_NANOLLAVA_TORCH_REVISION"] = NANOLLAVA_REVISION.strip()
    # Those weights ship an auto_map, so loading them runs model code from the
    # pinned snapshot. It is off unless you turn it on deliberately.
    os.environ["FORECAST_NANOLLAVA_TRUST_REMOTE_CODE"] = "true"

if ANTHROPIC_API_KEY.strip():
    os.environ["INSIGHT_PROVIDER"] = "anthropic"
    os.environ["INSIGHT_CLOUD_ENABLED"] = "true"
    os.environ["ANTHROPIC_API_KEY"] = ANTHROPIC_API_KEY.strip()
    os.environ["INSIGHT_ANTHROPIC_MODEL"] = "claude-sonnet-4-5-20250929"
elif INSIGHT_REVISION.strip():
    os.environ["INSIGHT_PROVIDER"] = "torch-local"
    os.environ["INSIGHT_LOCAL_MODEL"] = "Qwen/Qwen2.5-7B-Instruct"
    os.environ["INSIGHT_LOCAL_MODEL_REVISION"] = INSIGHT_REVISION.strip()

if HF_TOKEN.strip():
    os.environ["HF_TOKEN"] = HF_TOKEN.strip()

for key in ("INSIGHT_ASR_MODEL_REVISION", "FORECAST_NANOLLAVA_TORCH_REVISION"):
    print(f"{key:<38}", os.environ.get(key) or "not pinned -> lane unavailable")
print(f"{'insight provider':<38}", os.environ.get("INSIGHT_PROVIDER", "mlx-local (unavailable here)"))

### Fetching the weights

A pinned lane resolves its snapshot from the local cache and will not download whatever the
hub is serving today. Fetch each one once, then it runs offline.

In [ ]:
from huggingface_hub import snapshot_download
import os

for repo, key in (
    ("openai/whisper-large-v3-turbo", "INSIGHT_ASR_MODEL_REVISION"),
    ("qnguyen3/nanoLLaVA-1.5", "FORECAST_NANOLLAVA_TORCH_REVISION"),
    ("Qwen/Qwen2.5-7B-Instruct", "INSIGHT_LOCAL_MODEL_REVISION"),
):
    revision = os.environ.get(key, "")
    if not revision:
        print(f"skipping {repo}: not pinned")
        continue
    print(f"fetching {repo} @ {revision[:12]} ...")
    snapshot_download(repo_id=repo, revision=revision)
    print("  cached")

## 5. A test clip (optional)

Fourteen seconds, deliberately silent for the first 1.4. That makes the flagged
opening-silence check reproducible, and makes the recut's effect measurable.
Skip this and upload your own clip instead.

In [ ]:
!./scripts/make-demo-clip.sh /content/demo-clip.mp4

## 6. Launch

Starts the API in this process and serves the Gradio interface. `share=True` prints a
public `*.gradio.live` URL — open it in any browser. It stays alive while this cell runs.

In [ ]:
import os, sys
sys.path.insert(0, os.getcwd())

os.environ["SIGNALFRAME_SHARE"] = "true"

from colab.gradio_app import build_interface, start_backend_in_background

start_backend_in_background()
build_interface().launch(share=True, server_name="0.0.0.0", server_port=7860)

---

## What to do in the interface

1. **Analyse** — upload the clip. You will see each branch report available, or unavailable
   with the reason it names. Nothing is substituted for a missing branch.
2. **Hook readout** — the first three seconds as a timeline plus five checks. No model is
   involved. A check that cannot be measured says `not measured`, never `clear`.
3. **Hook Doctor** — cited notes, if you set a key in step 4. Every sentence carries the
   evidence it came from, and the artifact is rejected whole if any citation does not resolve.
4. **Compare cuts** — analyse two cuts, then see which measured signals moved.
5. **Recut** — trim the dead air, download, and analyse the result. The opening-silence check
   should move from flagged to clear, and the silent-window fraction toward zero.

That loop — measure, change one thing, measure again — is the whole product.

---

## Optional: the TRIBE v2 cortical lane

Only worth it if you specifically want the cortical prediction. It is a large, gated install,
and it does **not** improve hook advice: cortical output describes predicted average-subject
BOLD, and this project refuses to treat it as attention, engagement, or performance.

**Before running this**, you must have a Hugging Face account that has accepted the TRIBE v2
terms. The weights are **CC BY-NC 4.0** — non-commercial unless you obtain separate
permission. The checkpoint is hash-verified before load and refused if it does not match.

In [ ]:
# Heavy: torch, the pinned tribev2 package, and multi-GB weights.
# Only run this if you have accepted the gated terms and set HF_TOKEN above.
import os

assert os.environ.get("HF_TOKEN"), "Set HF_TOKEN in step 4 first."

!pip install -q torch torchvision
!pip install -q timm einops transformers
!pip install -q -r backend/requirements.txt

# Colab is the deployment this path was designed for: CUDA, and the full
# trimodal mode rather than the macOS vision-only ablation.
os.environ["TRIBE_MODEL_ID"] = "facebook/tribev2"
os.environ["TRIBE_MODEL_REVISION"] = "f894e783020944dcd96e5568550afe2aa9743f9f"
os.environ["TRIBE_CHECKPOINT_NAME"] = "best.ckpt"
os.environ["TRIBE_DEVICE"] = "cuda"
os.environ["TRIBE_VIDEO_DEVICE"] = "cuda"
os.environ["TRIBE_VIDEO_PRECISION"] = "fp32"
os.environ["TRIBE_INFERENCE_MODE"] = "full"

print("Configured. Restart the launch cell for it to take effect.")
print("If the checkpoint digest does not match the audited value, the load is refused")
print("on purpose — that is the integrity check, not a bug.")

---

## Where things are kept

Everything the app writes lives under `backend/.runtime/` in this Colab session and
**disappears when the runtime is recycled**:

- `backend/.runtime/forecast/results/` — published evidence
- `backend/.runtime/insight/artifacts/` — generated insight artifacts and the exact
  evidence each one cites
- `backend/.runtime/insight/rejections/` — refusals, including the offending sentence

Colab is a shared, ephemeral machine. Treat anything you upload as leaving your control:
for client footage or anything sensitive, run it locally with `./scripts/start-mac.sh`
instead, where nothing leaves the machine unless you switch on the remote provider.